# Clinical Healthcare Analytics: Early Cardio-Metabolic Disease Risk Prediction
**Author:** Naveen Kakarla  
**Email:** kakarlanaveen19@gmail.com  
**Program:** BharatCares AI & Data Analytics Masterclass Capstone  
**Mentors:** Himanshu Souda & Kartik Hooda (BharatCares)  
**United Nations SDG Alignment:** Goal 3 - Good Health and Well-being  

---
## 1. Project Background & Clinical Motivation
Cardiovascular and metabolic diseases represent the leading cause of non-communicable mortality worldwide. Early detection using non-invasive clinical biomarkers (fasting blood glucose, BMI, blood pressure, cholesterol, and lifestyle factors) empowers healthcare providers to intervene proactively.

In this project, we implement an end-to-end Machine Learning pipeline to predict high-risk disease onset with high recall and precision.


In [1]:
# Import Core Scientific & Machine Learning Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

print("Data Science and Healthcare ML environment initialized successfully!")


In [2]:
# 2. Loading the Clinical Patient Dataset
# Load dataset (or load from local file clinical_cardio_risk_data.csv)
df = pd.read_csv('clinical_cardio_risk_data.csv')
print("Dataset Shape:", df.shape)
df.head()


In [3]:
# 3. Exploratory Data Analysis (EDA) & Summary Statistics
print("Summary of Patient Biomarkers:")
display(df.describe().round(2))
print("\nCheck Missing Values:")
print(df.isnull().sum())
print("\nClass Distribution (0 = Low Risk, 1 = High Risk):")
print(df['HighRisk'].value_counts(normalize=True).round(3))


In [4]:
# 4. Clinical Visualizations (Distributions & Biomarker Correlations)
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=df, x='Glucose_mg_dl', hue='HighRisk', kde=True, palette='Set2')
plt.title('Fasting Glucose by Risk Level', fontsize=12)

plt.subplot(1, 3, 2)
sns.boxplot(data=df, x='HighRisk', y='Cholesterol_mg_dl', palette='Set1')
plt.title('Cholesterol vs High Risk Diagnosis', fontsize=12)

plt.subplot(1, 3, 3)
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', cbar=False)
plt.title('Correlation Matrix of Biomarkers', fontsize=12)

plt.tight_layout()
plt.savefig('healthcare_eda_charts.png', dpi=300)
plt.show()


In [5]:
# 5. Data Splitting & Feature Standardization
X = df.drop(columns=['HighRisk'])
y = df['HighRisk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")


In [6]:
# 6. Multi-Model Benchmark & Cross Validation
classifiers = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, random_state=42)
}

results = []
trained_models = {}

for name, clf in classifiers.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    y_proba = clf.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    trained_models[name] = clf
    results.append({
        'Classifier': name,
        'Accuracy': round(acc * 100, 2),
        'Precision': round(prec * 100, 2),
        'Recall (Sensitivity)': round(rec * 100, 2),
        'F1-Score': round(f1 * 100, 2),
        'ROC-AUC': round(auc, 4)
    })

leaderboard = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
print("Healthcare Model Evaluation Leaderboard:")
display(leaderboard)


In [7]:
# 7. Detailed Diagnostics for Best Model (Gradient Boosting)
best_model = trained_models['Gradient Boosting']
y_pred_best = best_model.predict(X_test_scaled)
y_prob_best = best_model.predict_proba(X_test_scaled)[:, 1]

print("Classification Report:\n", classification_report(y_test, y_pred_best))

plt.figure(figsize=(12, 5))
# Confusion Matrix
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Low Risk', 'High Risk'], 
            yticklabels=['Low Risk', 'High Risk'])
plt.title('Clinical Confusion Matrix', fontsize=12)
plt.ylabel('Actual Patient Status')
plt.xlabel('Predicted Status')

# ROC Curve
plt.subplot(1, 2, 2)
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_score(y_test, y_prob_best):.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC)', fontsize=12)
plt.legend(loc="lower right")

plt.tight_layout()
plt.savefig('healthcare_model_performance.png', dpi=300)
plt.show()

# Feature Importance
feat_imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top Clinical Biomarker Drivers:")
print(feat_imp.round(4))
